#### **Dimenstion Table : dim_customer**

In [1]:
create or replace table Olist_Gold_Lakehouse.dim_customer as

with geolocation_table as (
    select
        geolocation_zip_code_prefix,
        avg(geolocation_lat) as geolocation_lat,
        avg(geolocation_lng) as geolocation_lng
    from Olist_Silver_Lakehouse.silver_olist_geolocation
    group by geolocation_zip_code_prefix
),

customer_latest as (
    select *
    from (
        select
            c.*,
            row_number() over (
            partition by c.customer_unique_id 
            order by c.customer_id desc   
            ) as rn
        from Olist_Silver_Lakehouse.silver_olist_customers c
    )
    where rn = 1
)

select
    row_number() over(order by customer_unique_id) as customer_key,
    customer_unique_id,
    customer_zip_code_prefix,
    customer_city as city,
    customer_state as state,
    g.geolocation_lat,
    g.geolocation_lng,
    case 
        when customer_unique_id = '-1' then 0 
        else 1 
    end as is_customer_valid,
    is_city_valid,
    is_state_valid,
    is_state_city_valid,
    case 
        when is_state_city_valid = 1 then 'Valid'
        else 'Invalid'
    end as location_status,
    current_timestamp() as created_date

from customer_latest c
left join geolocation_table g
on c.customer_zip_code_prefix = g.geolocation_zip_code_prefix;


StatementMeta(, a544d461-243a-4af5-9f71-02d74b3af644, 2, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [3]:
MERGE INTO Olist_Gold_Lakehouse.dim_customer AS target
USING (SELECT -1 AS customer_key) AS source
ON target.customer_key = source.customer_key

WHEN NOT MATCHED THEN
INSERT(
    customer_key,
    customer_unique_id,
    customer_zip_code_prefix,
    city,
    state,
    geolocation_lat,
    geolocation_lng,
    is_city_valid,
    is_state_valid,
    is_state_city_valid,
    location_status,
    created_date
) 

VALUES (
    -1,
    'UNKNOWN',
    NULL,
    'Unknown',
    'Unknown',
    NULL,
    NULL,
    NULL,
    NULL,
    NULL,
    'Unknown',
    CURRENT_TIMESTAMP()
);


StatementMeta(, ec145109-cdae-4f40-8527-86fb252dd1b2, 4, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

#### **Dimenstion Table : dim_seller**

In [25]:
create or replace table Olist_Gold_Lakehouse.dim_seller as
with gelocation_table as (
    select
    geolocation_zip_code_prefix,
    avg(geolocation_lat) as geolocation_lat,
    avg(geolocation_lng) as geolocation_lng
    from Olist_Silver_Lakehouse.silver_olist_geolocation
    group by geolocation_zip_code_prefix
)

select
row_number() over(order by s.seller_id) as seller_key,
s.seller_id,
s.seller_zip_code_prefix,
s.seller_city as city,
s.seller_state as state,
g.geolocation_lat,
g.geolocation_lng,
is_city_valid,
is_state_valid,
is_state_city_valid,
case when is_state_city_valid = 1 then 'Valid'
     else 'Invalid'
     end as location_status,
current_timestamp() as created_date
from Olist_Silver_Lakehouse.silver_olist_sellers s
left join gelocation_table g
on s.seller_zip_code_prefix = g.geolocation_zip_code_prefix


StatementMeta(, 5d368721-da44-451a-9a96-0c80b29648d6, 26, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

#### **Dimenstion Table : dim_product**

In [11]:
create or replace table Olist_Gold_Lakehouse.dim_product as
select
row_number() over(order by p.product_id) as product_key,
p.product_id,
coalesce(pc.product_category_name_english,p.product_category_name,'Unknown') as product_category,
p.product_name_lenght as product_name_length,
p.product_description_lenght as product_description_length,
p.product_photos_qty as product_photo_count,
p.product_weight_g as product_weight_grams,
p.product_length_cm,
p.product_height_cm,
p.product_width_cm,
p.is_product_name_length_valid,
p.is_product_weight_valid,
p.is_product_photos_valid,
p.is_product_dimension_valid,
current_timestamp() as created_date
from Olist_Silver_Lakehouse.silver_olist_products p
left join Olist_Silver_Lakehouse.silver_olist_product_category_translation pc
on p.product_category_name = pc.product_category_name

StatementMeta(, 111363f4-4cb3-4c51-8bed-8f0e2020f783, 12, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

#### **Dimention Table : dim_dates**

In [1]:
create or replace table Olist_Gold_Lakehouse.dim_dates as

with all_dates as (

    -- Only VALID purchase dates
    select cast(order_purchase_timestamp as date) as dt
    from Olist_Gold_Lakehouse.fact_orders
    where is_order_date_valid = 1

    union all

    -- Only VALID delivery dates
    select cast(order_delivered_customer_date as date)
    from Olist_Gold_Lakehouse.fact_orders
    where is_order_date_valid = 1

    union all

    -- Review dates
    select cast(review_creation_date as date)
    from Olist_Gold_Lakehouse.fact_order_review
    where review_creation_date is not null
),

date_range as (
    select
        min(dt) as start_date,
        greatest(max(dt), current_date()) as end_date
    from all_dates
    where dt is not null
)

select
    d as date,

    date_trunc('month', d) as MonthStart,
    date_format(d, 'MMM yyyy') as YearMonth,
    year(d)*100 + month(d) as YearMonthSort,

    year(d) AS year,
    concat('Q', quarter(d)) as quarter,
    concat(year(d), '-Q', quarter(d)) as YearQuarter,

    month(d) AS MonthNumber,
    date_format(d, 'MMM') as MonthName,

    day(d) AS day,
    weekofyear(d) as WeekNumber,

    date_format(d, 'EEEE') AS weekday,

    case 
        when date_format(d, 'E') in ('Sat','Sun') then 1 
        else 0 
    end as IsWeekend,

    case 
        when date_format(d, 'E') in ('Sat','Sun') then 'Weekend' 
        else 'Weekday' 
    end as DayType,

    current_timestamp() as created_date

from date_range
lateral view explode(sequence(start_date, end_date, interval 1 day)) t AS d;

StatementMeta(, 03af3ebb-b23b-4380-825b-82e9e432528a, 2, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

#### **Fact Table : fact_order_items**

In [5]:
create or replace table Olist_Gold_Lakehouse.fact_order_items as

with payment_agg as (
    select 
        order_id,
        min(payment_key) as payment_key
    from Olist_Gold_Lakehouse.fact_order_payment
    group by order_id
),

review_agg as (
    select 
        order_id,
        min(review_key) as review_key
    from Olist_Gold_Lakehouse.fact_order_review
    group by order_id
),

cte as (
select
    o.order_id,
    oi.order_item_id,
    coalesce(c.customer_key, -1) as customer_key,
    coalesce(c.customer_unique_id, '-1') as customer_unique_id,
    coalesce(p.product_key,-1) as product_key,
    coalesce(s.seller_key,-1) as seller_key,
    coalesce(r.review_key,-1) as review_key,
    coalesce(po.payment_key,-1) as payment_key,

    oi.price,
    oi.freight_value,

    case 
        when oi.price is not null and oi.freight_value is not null 
        then oi.price + oi.freight_value
        else null
    end as total_order_value,

    o.order_status,
    o.order_purchase_timestamp,
    cast(o.order_purchase_timestamp as date) as order_purchase_date,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,

    oi.is_delivered_price_missing,
    oi.is_delivered_freight_value_missing,
    o.is_late_delivery,

    case 
        when po.payment_key is null then 0
        else 1
    end as is_payment_valid,

    oi.is_refunded,

    case 
        when order_status = 'delivered' then 1 
        else 0
    end as is_order_completed,

    is_invalid_canceled_delivery,
    is_order_date_valid,

    current_timestamp() as created_date

from Olist_Silver_Lakehouse.silver_olist_order_items oi
left join Olist_Silver_Lakehouse.silver_olist_orders o
    on o.order_id = oi.order_id
left join Olist_Gold_Lakehouse.dim_customer c
    on o.customer_unique_id = c.customer_unique_id
left join Olist_Gold_Lakehouse.dim_seller s
    on oi.seller_id = s.seller_id
left join Olist_Gold_Lakehouse.dim_product p 
    on oi.product_id = p.product_id
left join review_agg r
    on oi.order_id = r.order_id
left join payment_agg po
    on oi.order_id = po.order_id
)

select 
    order_id,
    order_item_id,
    customer_key,
    customer_unique_id,
    product_key,
    seller_key,
    review_key,
    payment_key,
    price,
    freight_value,
    total_order_value,
    order_status,
    order_purchase_timestamp,
    order_purchase_date,
    order_approved_at,
    order_delivered_carrier_date,
    order_delivered_customer_date,
    order_estimated_delivery_date,
    is_payment_valid,
    is_refunded,
    is_order_completed,
    is_invalid_canceled_delivery,
    is_order_date_valid,
    created_date,

    case 
        when is_delivered_price_missing = 0
         and is_delivered_freight_value_missing = 0
         and is_order_date_valid = 1
         and is_invalid_canceled_delivery = 0
        then 1
        else 0
    end as is_valid_business_record,

    case 
        when order_status = 'delivered'
         and is_delivered_price_missing = 0
         and is_delivered_freight_value_missing = 0
         and is_order_date_valid = 1
         and is_invalid_canceled_delivery = 0
        then 1
        else 0
    end as is_valid_delivered_order

from cte;

StatementMeta(, 9f304d98-cb0c-4be2-9d87-1df98d221906, 6, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

#### **Fact Table : fact_orders**

In [2]:
create or replace table Olist_Gold_Lakehouse.fact_orders as

with order_item_agg as (
    select
        order_id,

        -- Refund logic
        max(is_refunded) as has_refund,
        min(is_refunded) as is_fully_refunded,

        -- Completion
        max(is_order_completed) as is_order_completed,

        -- Data quality (STRICT → all items must be valid)
        min(is_order_date_valid) as is_order_date_valid,
        max(is_valid_business_record) as is_valid_business_record,
        max(is_valid_delivered_order) as is_valid_delivered_order,

        -- Payment (loose logic → at least one valid payment)
        max(is_payment_valid) as is_payment_valid,

        sum(
            case 
            when is_valid_business_record = 1 
            and is_order_completed = 1 
            then price 
            else 0 
            end
        ) as total_price,

        sum(
            case 
            when is_valid_business_record = 1 
            and is_order_completed = 1 
            then freight_value 
            else 0 
            end
        ) as total_freight_value,

        sum(
            case 
            when is_valid_business_record = 1 
            and is_order_completed = 1 
            then price + freight_value
            else 0 
            end
        ) as total_order_value,

        sum(
            case 
            when is_refunded = 1 
            then price 
            else 0 
            end
        ) as refunded_price,

count(order_item_id) as item_count

    from Olist_Gold_Lakehouse.fact_order_items
    group by order_id
)

,customer_cohort as (
    select
        customer_unique_id,
        date_trunc('month', min(order_purchase_timestamp)) as cohort_month
    from Olist_Silver_Lakehouse.silver_olist_orders
    group by customer_unique_id
)

,order_sequence as (
    select
        o.order_id,
        o.customer_unique_id,
        row_number() over (
            partition by o.customer_unique_id
            order by o.order_purchase_timestamp
        ) as order_number
    from Olist_Silver_Lakehouse.silver_olist_orders o
)

select 
    o.order_id,
    seq.order_number,
    o.customer_unique_id,
    coalesce(d.customer_key, -1) as customer_key,
    
    coalesce(oi.total_price, 0)         as total_price,
    coalesce(oi.total_freight_value, 0) as total_freight_value,
    coalesce(oi.total_order_value, 0)   as total_order_value,
    coalesce(oi.refunded_price, 0)      as refunded_price,
    oi.item_count,

    -- Order lifecycle
    o.order_status,
    o.order_purchase_timestamp,
    cast(o.order_purchase_timestamp as date) as order_purchase_date,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,

    -- Business flags
    o.is_late_delivery,

    coalesce(oi.is_order_completed, 0) as is_order_completed,
    coalesce(oi.is_order_date_valid, 0) as is_order_date_valid,
    coalesce(oi.is_valid_business_record, 0) as is_valid_business_record,
    coalesce(oi.is_valid_delivered_order, 0) as is_valid_delivered_order,
    coalesce(oi.is_payment_valid, 0) as is_payment_valid,

    -- Refund logic
    coalesce(oi.has_refund, 0) as has_refund,
    coalesce(oi.is_fully_refunded, 0) as is_fully_refunded,
    
    case 
        when coalesce(oi.has_refund, 0) = 0 then 'No Refund'
        when coalesce(oi.is_fully_refunded, 0) = 1 then 'Fully Refunded'
        else 'Partially Refunded'
    end as refund_status,

    -- Cohort analysis
    cc.cohort_month,

    cast(
        months_between(
            date_trunc('month', cast(o.order_purchase_timestamp as date)),
            cc.cohort_month
        ) as int
    ) as months_since_first_purchase,

    current_timestamp() as created_date

from Olist_Silver_Lakehouse.silver_olist_orders o

left join Olist_Gold_Lakehouse.dim_customer d
    on o.customer_unique_id = d.customer_unique_id

left join customer_cohort cc
    on o.customer_unique_id = cc.customer_unique_id

left join order_item_agg oi
    on o.order_id = oi.order_id

left join order_sequence seq
    on o.order_id = seq.order_id;

StatementMeta(, 5899aa62-2af4-4992-a6db-909ba226c909, 3, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

#### **Fact Table : fact_order_payment**

In [4]:
create or replace table fact_order_payment as
select
row_number() over(order by payment_sequential) as payment_key,
p.order_id,
p.payment_sequential,
p.payment_type,
p.payment_installments,
p.payment_value,
o.order_purchase_timestamp,
cast(o.order_purchase_timestamp as date) as order_purchase_date,
current_timestamp() as created_date
from Olist_Silver_Lakehouse.silver_olist_order_payments p
left join Olist_Silver_Lakehouse.silver_olist_orders o
    on p.order_id = o.order_id;

StatementMeta(, 9df5b076-e10f-44de-b343-bf1fafabba1f, 5, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

#### **Fact Table : fact_order_review**

In [18]:
create or replace table Olist_Gold_Lakehouse.fact_order_review as
select 
row_number() over(order by review_id) as review_key,
review_id,
order_id,
review_score,
review_comment_title,
review_comment_message,
review_creation_date,
review_answer_timestamp,
is_review_score_valid,
is_answer_after_review,
is_review_with_comment,
current_timestamp() as created_date
from Olist_Silver_Lakehouse.silver_olist_order_reviews

StatementMeta(, b500c9d1-1639-4335-9480-3cfa4f52a553, 19, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [1]:
create or replace table Olist_Gold_Lakehouse.customer_summary as
select
c.customer_unique_id,
count(distinct o.order_id) AS total_orders,
case when count(distinct o.order_id) = 1 then 'New'
     when count(distinct o.order_id) <= 3 then 'Repeat'
     else 'Loyal'
end as customer_segment
from Olist_Gold_Lakehouse.fact_orders o
left join Olist_Gold_Lakehouse.dim_customer c
on o.customer_key = c.customer_key
group by c.customer_unique_id;

StatementMeta(, 6f4d93d2-0d5e-432d-93e5-a1451226aeb5, 2, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [2]:
CREATE TABLE IF NOT EXISTS gold_data_quality_log (
    run_id STRING,
    run_type STRING,
    table_name STRING,
    check_name STRING,
    status STRING,
    percentage DOUBLE,
    record_count BIGINT,
    run_timestamp TIMESTAMP,
    run_date DATE
)
USING DELTA
PARTITIONED BY (run_date);

StatementMeta(, 9f743587-1793-404d-adb1-690e6b956a71, 3, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [1]:
select * from gold_data_quality_log

StatementMeta(, ee037107-4c05-43c0-8155-1c774976a74d, 2, Finished, Available, Finished, False)

<Spark SQL result set with 95 rows and 9 fields>

In [51]:
with base as(
select
cast(date_trunc('month',min(order_purchase_date)) as date) as min_date,
cast(date_trunc('month',max(order_purchase_date)) as date) as max_date
from fact_order_items
)
, date_seq as(
select explode(sequence(min_date,max_date,interval 1 month)) as month_start
from base
)
, total_sales as(
select
cast(date_trunc('month',order_purchase_date) as date) as month_start,
SUM(price + freight_value) as total_sales
from fact_order_items
where is_valid_business_record = 1
and is_refunded = 0
group by month_start
)
,final as(
select
cast(date_trunc('month',m.month_start) as date) as month_start,
coalesce(s.total_sales,0) as total_sales,
lag(coalesce(s.total_sales,0)) over(order by m.month_start) as previous_month_sale
from date_seq m
left join total_sales s
on m.month_start = s.month_start
)
select
date_format(month_start,'MMM yyyy') as month,
total_sales,
case when previous_month_sale = 0 then null 
else
round(((total_sales - previous_month_sale)*100 / previous_month_sale),2)
end as mom_growth
from final



StatementMeta(, e13ac806-bd8c-425b-9d0e-933a01be8d89, 52, Finished, Available, Finished, False)

<Spark SQL result set with 115 rows and 3 fields>

In [49]:
with cte as (
select 
date_trunc('month', d.date) as curr_month,
sum(coalesce(oi.total_order_value,0)) as current_month_sales
from dim_dates d 
left join fact_order_items oi
on d.date = cast(oi.order_purchase_timestamp as date)
and oi.is_valid_business_record = 1
group by curr_month
)
, cte2 as(
select
*,
lag(curr_month) over(order by curr_month) as prev_month,
lag(current_month_sales) over(order by curr_month) as previous_month_sales
from cte
)
select
date_format(curr_month, 'MMM yyyy') as curr_month,
current_month_sales,
date_format(prev_month, 'MMM yyyy') as prev_month,
previous_month_sales,
concat(round((current_month_sales - previous_month_sales)*100 / nullif(previous_month_sales,0),2),'%') as mom_growth
from cte2

StatementMeta(, e13ac806-bd8c-425b-9d0e-933a01be8d89, 50, Finished, Available, Finished, False)

<Spark SQL result set with 116 rows and 5 fields>

In [55]:
with cte as (
select 
date_trunc('month', d.date) as month,
sum(coalesce(oi.total_order_value,0)) as total_sales
from dim_dates d 
left join fact_order_items oi
on d.date = cast(oi.order_purchase_timestamp as date)
and oi.is_valid_business_record = 1
group by month
)
, cte2 as(
select
*,
sum(total_sales) over(order by month rows between unbounded preceding and current row) as rolling_total
from cte
)
select
date_format(month, 'MMM yyyy') as curr_month,
total_sales,
rolling_total
from cte2

StatementMeta(, e13ac806-bd8c-425b-9d0e-933a01be8d89, 56, Finished, Available, Finished, False)

<Spark SQL result set with 116 rows and 3 fields>